# Course 10: Intermediate Git

This notebook records core concepts, branching workflows, branch comparisons, merging, merge conflict resolution, and remote repository synchronization (`clone`, `fetch`, `pull`, `push`) from the **Intermediate Git** course on DataCamp (instructed by George Boorman).

---
## Chapter 1: Introduction to Branches

### 1.1 Understanding & Navigating Branches
* **Branch**: An independent pointer/version of a repository that lets you isolate new feature development or bug fixes from the stable production branch (`main`).
* **Why Use Branches?**:
  * Keep the live/production branch (`main`) clean and working as expected while testing experimental changes safely.
  * Enable multiple developers to work concurrently on separate features (`ai-assistant`, `bug-fix`) without interfering with each other.
  * Compare repository states across branches before integrating changes.
* **Core Branch Navigation Commands**:
  * `git branch` — List all local branches (`*` marks the currently checked-out branch).
  * `git switch main` — Switch to an existing branch (`main`).
  * `git branch speed-test` + `git switch speed-test` — Create a new branch ("branching off") and move to it.
  * `git switch -c speed-test` — Create a new branch and switch to it in a single command.

### 1.2 Comparing, Renaming, and Deleting Branches
* **Comparing Branches**: `git diff main summary-statistics` shows file differences between two branches (use `Space` to advance and `q` to exit long pager outputs).
* **Renaming a Branch**: `git branch -m feature_dev chatbot` renames `feature_dev` to `chatbot`.
* **Deleting a Branch**:
  * `git branch -d chatbot` — Safely deletes a branch **only if** its changes have already been merged.
  * `git branch -D chatbot` — Force-deletes an **unmerged** branch (use with caution—recovery is difficult).

### 1.3 Merging Branches
* **Source vs. Destination**:
  * **Source**: The branch containing changes you want to integrate (e.g., `ai-assistant`).
  * **Destination**: The branch receiving the changes (e.g., `main`).
  * **Parent Commits**: The latest commits on each branch being merged.
* **Merging Commands**:
  * Standard two-step workflow:
    ```bash
    git switch main
    git merge ai-assistant
    ```
  * Direct two-argument syntax from any branch: `git merge ai-assistant main` (`git merge <source> <destination>`).
* **Fast-Forward Merge**: When `main` has no divergent commits since `ai-assistant` branched off, Git simply advances `main`'s pointer forward (`Fast-forward`) with a linear commit history.

In [ ]:
import os
import shutil
import subprocess
import tempfile

def run_git(cmd, cwd):
    if shutil.which("git") is None:
        print(f"$ {' '.join(cmd)}")
        print("[Note: `git` binary not detected in this runtime; execute locally with Git installed.]")
        print("-" * 60)
        return None
    result = subprocess.run(cmd, cwd=cwd, text=True, capture_output=True)
    print(f"$ {' '.join(cmd)}")
    if result.stdout.strip():
        print(result.stdout.strip())
    if result.stderr.strip():
        print(result.stderr.strip())
    print("-" * 60)
    return result

def git_exec(cmd, cwd):
    if shutil.which("git") is not None:
        subprocess.run(cmd, cwd=cwd, check=True, capture_output=True)

with tempfile.TemporaryDirectory() as tmpdir:
    git_exec(["git", "init", "-b", "main"], tmpdir)
    git_exec(["git", "config", "user.name", "George Boorman"], tmpdir)
    git_exec(["git", "config", "user.email", "george@datacamp.com"], tmpdir)

    # Initial commit on main
    os.makedirs(os.path.join(tmpdir, "source"), exist_ok=True)
    with open(os.path.join(tmpdir, "README.md"), "w") as f:
        f.write("# DataCamp Project\nMain production branch.\n")
    git_exec(["git", "add", "."], tmpdir)
    git_exec(["git", "commit", "-m", "Initial commit on main"], tmpdir)

    # 1. Create and switch to feature_dev, then rename to ai-assistant
    run_git(["git", "switch", "-c", "feature_dev"], tmpdir)
    run_git(["git", "branch", "-m", "feature_dev", "ai-assistant"], tmpdir)
    run_git(["git", "branch"], tmpdir)

    # 2. Add feature work on ai-assistant
    with open(os.path.join(tmpdir, "source", "main.py"), "w") as f:
        f.write("def recommend_courses(prompt):\n    return ['Introduction to Git', 'Intermediate Git']\n")
    git_exec(["git", "add", "."], tmpdir)
    git_exec(["git", "commit", "-m", "Add AI assistant recommender module"], tmpdir)

    # 3. Compare main vs ai-assistant
    run_git(["git", "diff", "main", "ai-assistant"], tmpdir)

    # 4. Switch to main, fast-forward merge ai-assistant, and delete merged branch
    run_git(["git", "switch", "main"], tmpdir)
    run_git(["git", "merge", "ai-assistant"], tmpdir)
    run_git(["git", "branch", "-d", "ai-assistant"], tmpdir)


### Chapter 1: Key Takeaways & Actionable Step-by-Step Process

#### Key Takeaways
* Isolate every feature or bug fix on its own dedicated branch (`git switch -c <branch-name>`) so `main` always remains deployable.
* Always check `git branch` and `git diff main <branch>` before merging to confirm you are on the destination branch and understand all incoming changes.
* Clean up completed feature branches with `git branch -d <branch>` after merging (reserving `-D` strictly for intentionally discarded unmerged experiments).

---
## Chapter 2: Merge Conflicts & Collaborating with Remotes

### 2.1 Resolving Merge Conflicts
* **When Conflicts Happen**: Two branches modify the same lines (or both add conflicting versions) of a file, and Git cannot automatically determine which version to keep (`CONFLICT (content)` / `CONFLICT (add/add)`).
* **Conflict Marker Syntax**:
  ```text
  # Contents and usage
  This repo contains source code for the DataCamp website.
  <<<<<<< HEAD
  It is for internal use only and external access is prohibited.
  =======
  It also contains source code for an AI-Assistant (recommendation system).
  >>>>>>> documentation
  ```
  * `<<<<<<< HEAD` to `=======`: Content in your current branch (e.g., `main`).
  * `=======` to `>>>>>>> documentation`: Content from the incoming branch (`documentation`).
* **Resolution Workflow**:
  1. Open the conflicted file (`nano README.md`).
  2. Edit the text to retain the desired combination of lines and **delete all three conflict marker lines** (`<<<<<<<`, `=======`, `>>>>>>>`).
  3. Save and exit (`Ctrl + O`, `Enter`, `Ctrl + X` in `nano`).
  4. Stage and commit the resolved file:
     ```bash
     git add README.md
     git commit -m "Resolving README.md conflict"
     ```

### 2.2 Working with Remote Repositories
* **Local vs. Remote Repo**: Remote repositories reside on hosting platforms (GitHub, GitLab, Bitbucket) or shared file paths, providing off-site backup and multi-user collaboration.
* **Cloning Repositories**:
  * `git clone /home/george/repo [new_repo]` — Clone a local repository path.
  * `git clone https://github.com/datacamp/project` — Clone a remote repository and automatically configure its remote alias as `origin`.
* **Inspecting & Adding Remotes**:
  * `git remote` — List configured remote names.
  * `git remote -v` — Display remote names alongside their `(fetch)` and `(push)` URLs.
  * `git remote add george https://github.com/george_datacamp/repo` — Register an additional remote alias.

### 2.3 Fetching & Pulling from Remotes
* **Fetching (`git fetch`)**: Downloads new commits and branches from a remote (`git fetch origin` or `git fetch origin main`) into remote-tracking references without modifying your local working directory. Merge afterward with `git merge origin` (or `git merge origin/main`).
* **Pulling (`git pull`)**: Combines `git fetch` + `git merge` in a single step:
  * `git pull origin` — Fetch and merge the remote's default branch into your current local branch.
  * `git pull origin dev` — Fetch and merge the remote `dev` branch into your current local branch.
* **Safety Rule**: Commit or stash local changes before pulling; otherwise Git aborts (`error: Your local changes to the following files would be overwritten by merge`).

### 2.4 Pushing to Remotes & Avoiding Non-Fast-Forward Rejections
* **Pushing Changes (`git push`)**:
  * `git push origin main` — Upload committed local changes from `main` to `origin`.
  * `git push origin hotfix` — Push a brand-new local branch `hotfix` to `origin` and establish upstream tracking.
* **Non-Fast-Forward Push Rejection**: If teammates pushed commits to `origin/main` that you haven't pulled yet, `git push origin main` fails with `! [rejected] main -> main (non-fast-forward)`.
* **Recommended Collaboration Loop**:
  1. `git pull origin main` (or `git pull --no-edit origin main` when confident no manual merge message edit is needed).
  2. Resolve any merge conflicts if prompted.
  3. `git push origin main`.

In [ ]:
import os
import shutil
import subprocess
import tempfile

def run_git(cmd, cwd):
    if shutil.which("git") is None:
        print(f"$ {' '.join(cmd)}")
        print("[Note: `git` binary not detected in this runtime; execute locally with Git installed.]")
        print("-" * 60)
        return None
    result = subprocess.run(cmd, cwd=cwd, text=True, capture_output=True)
    print(f"$ {' '.join(cmd)}")
    if result.stdout.strip():
        print(result.stdout.strip())
    if result.stderr.strip():
        print(result.stderr.strip())
    print("-" * 60)
    return result

def git_exec(cmd, cwd):
    if shutil.which("git") is not None:
        subprocess.run(cmd, cwd=cwd, check=True, capture_output=True)

with tempfile.TemporaryDirectory() as tmpdir:
    # Part A: Create a bare "remote" repo and clone a local working copy
    remote_repo = os.path.join(tmpdir, "origin_project.git")
    git_exec(["git", "init", "--bare", "-b", "main", remote_repo], tmpdir)

    alice_repo = os.path.join(tmpdir, "alice_local")
    os.makedirs(alice_repo, exist_ok=True)
    run_git(["git", "clone", remote_repo, alice_repo], tmpdir)
    git_exec(["git", "config", "user.name", "Alice"], alice_repo)
    git_exec(["git", "config", "user.email", "alice@datacamp.com"], alice_repo)

    # Verify remote URLs with git remote -v
    run_git(["git", "remote", "-v"], alice_repo)

    # Alice creates initial README.md and pushes to origin main
    readme_alice = os.path.join(alice_repo, "README.md")
    with open(readme_alice, "w") as f:
        f.write("# Contents and usage\nThis repo contains source code for the DataCamp website.\n")
    git_exec(["git", "add", "README.md"], alice_repo)
    git_exec(["git", "commit", "-m", "Initial README"], alice_repo)
    run_git(["git", "push", "origin", "main"], alice_repo)

    # Part B: Simulate a merge conflict between main and documentation branch
    git_exec(["git", "switch", "-c", "documentation"], alice_repo)
    with open(readme_alice, "w") as f:
        f.write(
            "# Contents and usage\n"
            "This repo contains source code for the DataCamp website.\n"
            "It also contains source code for an AI-Assistant (recommendation system).\n"
        )
    git_exec(["git", "commit", "-am", "Document AI assistant"], alice_repo)

    # Conflicting edit on main
    git_exec(["git", "switch", "main"], alice_repo)
    with open(readme_alice, "w") as f:
        f.write(
            "# Contents and usage\n"
            "This repo contains source code for the DataCamp website.\n"
            "It is for internal use only, external access is prohibited.\n"
        )
    git_exec(["git", "commit", "-am", "Add internal access notice"], alice_repo)

    # Attempt merge -> triggers CONFLICT
    run_git(["git", "merge", "documentation"], alice_repo)

    # Resolve conflict by combining both clauses and removing conflict markers
    with open(readme_alice, "w") as f:
        f.write(
            "# Contents and usage\n"
            "This repo contains source code for the DataCamp website.\n"
            "It also contains source code for an AI-Assistant (recommendation system).\n"
            "It is for internal use only, external access is prohibited.\n"
        )
    run_git(["git", "add", "README.md"], alice_repo)
    run_git(["git", "commit", "-m", "Resolving README.md conflict"], alice_repo)

    # Push resolved main and a new hotfix branch to origin
    run_git(["git", "push", "origin", "main"], alice_repo)
    git_exec(["git", "switch", "-c", "hotfix"], alice_repo)
    run_git(["git", "push", "origin", "hotfix"], alice_repo)


### Chapter 2: Key Takeaways & Actionable Step-by-Step Process

#### Key Takeaways
* Merge conflicts are normal when parallel branches edit the same lines; resolve them systematically by removing all `<<<<<<<`, `=======`, and `>>>>>>>` markers, staging with `git add`, and committing.
* `git fetch` downloads remote updates safely without altering local files, whereas `git pull` fetches and immediately merges into your active branch.
* Always commit local work and run `git pull origin <branch>` before `git push origin <branch>` to prevent non-fast-forward rejections.

#### Complete Command Reference — Intermediate Git
| Command | Purpose / Effect |
| :--- | :--- |
| `git branch` | List all local branches (`*` denotes active branch) |
| `git switch <branch>` | Switch to an existing branch |
| `git switch -c <branch>` | Create a new branch and switch to it immediately |
| `git diff <branch1> <branch2>` | Compare file differences between two branches |
| `git branch -m <old> <new>` | Rename an existing branch |
| `git branch -d <branch>` / `-D` | Safely delete a merged branch (`-d`) or force-delete unmerged (`-D`) |
| `git merge <source>` | Merge `<source>` branch into the currently checked-out branch |
| `git clone <url_or_path> [dir]` | Copy a local or remote repository and configure `origin` |
| `git remote -v` | Display configured remote aliases and fetch/push URLs |
| `git remote add <name> <url>` | Add a new remote repository connection |
| `git fetch <remote> [branch]` | Download remote changes without merging into local branches |
| `git pull <remote> [branch]` | Fetch remote changes and merge them into the current branch |
| `git push <remote> <branch>` | Upload local branch commits (or create a new branch) on the remote |